# Lecture 5 Deep Learning I: Neural Network & PyTorch — จาก Random Forest สู่สมองกล

**Phase 3: Deep Learning เริ่มต้นที่นี่**

มี AI QC ที่ใช้ **Random Forest** ไปติดตั้งสายการผลิต CNC จริง — ทำงานได้ดี F1 = 0.97
จะตอบคำถามใหญ่: **ถ้า Random Forest เก่งขนาดนี้แล้ว ทำไมโลกทั้งใบถึงเห่อ Deep Learning? ทำไม ChatGPT / Midjourney ไม่ใช้ Random Forest?**

ผลลัพธ์การเรียนรู้:
- เข้าใจ Neural Network ตั้งแต่เซลล์เดียว (Perceptron) จนถึงหลายชั้น (Deep)
- รู้ว่า Activation Function คืออะไร ทำไมขาดไม่ได้
- สร้าง Neural Network ตัวแรกด้วย PyTorch ด้วยตัวเอง
- เทียบ NN กับ Random Forest บนงาน CNC QC จริง แล้วค้นพบความจริงที่หลายคนเข้าใจผิด
- อธิบายโครงสร้างและหลักการทำงานของ Neural Network (Input, Hidden, Output Layer) ได้
- เปรียบเทียบ Activation Functions ประเภทต่าง ๆ (ReLU, Sigmoid, Softmax) และเลือกใช้ได้อย่างเหมาะสม
- สร้าง Multilayer Perceptron (MLP) ด้วย TensorFlow/Keras/Pytorch ได้


### ขั้นตอนแรก: ติดตั้ง Font ภาษาไทยสำหรับ Matplotlib

In [ ]:
# ติดตั้ง font Sarabun (ภาษาไทย) สำหรับ matplotlib
!wget -q https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Regular.ttf -O /usr/share/fonts/truetype/Sarabun-Regular.ttf
!wget -q https://github.com/google/fonts/raw/main/ofl/sarabun/Sarabun-Bold.ttf -O /usr/share/fonts/truetype/Sarabun-Bold.ttf
!fc-cache -f

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
fm.fontManager.addfont('/usr/share/fonts/truetype/Sarabun-Regular.ttf')
fm.fontManager.addfont('/usr/share/fonts/truetype/Sarabun-Bold.ttf')
plt.rcParams['font.family'] = 'Sarabun'
plt.rcParams['axes.unicode_minus'] = False
print("ติดตั้ง font Sarabun เรียบร้อย")

### Imports หลักที่ใช้วันนี้

วันนี้มีของใหม่: **PyTorch** — ใน Google Colab ติดตั้งมาให้แล้ว แค่ `import torch` ได้เลย ไม่ต้องลงเอง

In [ ]:
# ========== Imports ==========
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch — พระเอกของ Phase 3 (Colab ติดตั้งมาให้แล้ว)
import torch
import torch.nn as nn

# Sklearn — ของเดิมจาก Phase 2 (ใช้เปรียบเทียบ + เตรียมข้อมูล)
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')

# ตั้ง seed ให้ผลรันซ้ำได้เหมือนเดิมทุกครั้ง (random_state=42 มาตรฐานของคอร์ส)
np.random.seed(42)
torch.manual_seed(42)

print("Imports พร้อมแล้ว")
print("PyTorch version:", torch.__version__)
print("มี GPU ใช้ไหม:", torch.cuda.is_available())

---

## ช่วงที่ 1:

Random Forest เก่งมากแล้ว — ทำไมยังต้องเรียน Neural Network?

### ทำไมถึงใช้ Deep Learning?

ลองคิดดู เครื่องมือ AI ที่ดังที่สุดในโลกตอนนี้ **ไม่มีตัวไหนใช้ Random Forest** เลย:

| เครื่องมือ | ทำอะไร | เบื้องหลัง |
|-----------|--------|-----------|
| ChatGPT | เข้าใจ + เขียนภาษา | Neural Network ขนาดยักษ์ |
| Midjourney | สร้างภาพจากข้อความ | Neural Network |
| GitHub Copilot | เขียนโค้ดให้ | Neural Network |
| Tesla Autopilot | มองถนน ขับรถ | Neural Network |

**ทำไม Random Forest ทำงานพวกนี้ไม่ได้?**

คำใบ้: Random Forest เก่งมากกับ **ข้อมูลตาราง** (ตัวเลขในคอลัมน์ เช่น vibration, temperature) — แต่มันไม่เคย "เห็น" ภาพ ไม่เคย "อ่าน" ประโยค

เฉลยเต็ม ๆ จะอยู่ตอนท้ายชั่วโมง (ช่วงที่ 6) หลังจากเราลองสร้าง Neural Network เองแล้ว

### แผนการเรียนวันนี้ — 5 ก้าว

1. **ช่วงที่ 2 — Perceptron:** เซลล์สมองตัวแรก (จริง ๆ เราเคยเจอมันแล้วใน Week 5!)
2. **ช่วงที่ 3 — ทำไมต้อง "ลึก" + Activation:** ทำไมเซลล์เดียวไม่พอ และทำไมต้องมี Activation
3. **ช่วงที่ 4 — PyTorch 101:** เครื่องมือมาตรฐานของวงการ (Tensor, Autograd, nn.Module)
4. **ช่วงที่ 5 — Training Loop:** หัวใจการเรียนรู้ของ NN (5 บรรทัดทอง)
5. **ช่วงที่ 6 — Mini Project:** เทียบ NN vs Random Forest บนงาน CNC จริง → เฉลยปริศนา

---

## ช่วงที่ 2: Perceptron — เซลล์สมองตัวแรก

ก่อนจะถึง Neural Network ขนาดยักษ์อย่าง ChatGPT เราต้องเข้าใจหน่วยที่เล็กที่สุดก่อน — **เซลล์เดียว** ที่เรียกว่า Perceptron

### 2.1 Analogy — เซลล์สมอง (neuron) กับ Perceptron

ในสมองคนเรามีเซลล์ประสาท (neuron) นับพันล้านตัว แต่ละตัวทำงานง่าย ๆ:

> รับสัญญาณเข้ามาหลายทาง → ถ้ารวมแล้วแรงพอ → ส่งสัญญาณต่อ

Perceptron เลียนแบบเซลล์นี้ ด้วย 4 ขั้นตอน:

1. **รับ input** หลายค่า (เช่น vibration, temperature)
2. **คูณน้ำหนัก (weight)** แต่ละ input — น้ำหนักมาก = สำคัญมาก
3. **บวกรวมกัน** + ค่าคงที่ (bias) ได้ผลรวมหนึ่งค่า
4. **ตัดสินใจ** ผ่าน activation — เกินเกณฑ์ = ส่งออก 1, ไม่เกิน = 0

สูตรหัวใจ: `output = activation(w1·x1 + w2·x2 + ... + b)`

In [ ]:
# ========== วาดภาพ Perceptron 1 เซลล์ ==========
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')

# input nodes
inputs = ['x1\n(vibration)', 'x2\n(temperature)', 'x3\n(current)']
for i, txt in enumerate(inputs):
    y = 3.5 - i*1.5
    ax.add_patch(plt.Circle((1, y), 0.35, color='#4C72B0', alpha=0.85))
    ax.text(1, y, txt, ha='center', va='center', color='white', fontsize=9)
    ax.annotate('', xy=(4.3, 2), xytext=(1.4, y),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    ax.text(2.6, (y+2)/2 + 0.15, f'w{i+1}', color='#C44E52', fontsize=11, fontweight='bold')

# neuron body
ax.add_patch(plt.Circle((4.7, 2), 0.7, color='#DD8452', alpha=0.9))
ax.text(4.7, 2.15, 'รวม', ha='center', va='center', color='white', fontsize=11, fontweight='bold')
ax.text(4.7, 1.75, 'Σ + b', ha='center', va='center', color='white', fontsize=9)

# activation
ax.add_patch(plt.Rectangle((6.0, 1.5), 1.3, 1.0, color='#55A868', alpha=0.9))
ax.text(6.65, 2, 'Activation', ha='center', va='center', color='white', fontsize=10)
ax.annotate('', xy=(6.0, 2), xytext=(5.4, 2), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))

# output
ax.annotate('', xy=(8.7, 2), xytext=(7.3, 2), arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.text(9.1, 2, 'output\n(0 / 1)', ha='center', va='center', fontsize=10, fontweight='bold')

ax.set_xlim(0, 10); ax.set_ylim(0, 4)
ax.set_title('Perceptron 1 เซลล์: รับ input -> คูณน้ำหนัก -> รวม -> ตัดสินใจ', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 2.2 ความลับ: Perceptron = Logistic Regression ที่เราเคยเรียน Supervied Learning!

ถ้า Perceptron รู้สึกคุ้น ๆ — เพราะเราเจอมันแล้วใน **Supervied Learning**!

**Logistic Regression** ที่เราใช้ทำ Classification ก็คือ Perceptron 1 เซลล์ ที่ใช้ Sigmoid เป็น activation :
- คูณน้ำหนักแต่ละ feature → บวกรวม → ผ่าน Sigmoid → ได้ความน่าจะเป็น 0-1

นั่นคือ: **Neural Network ที่เล็กที่สุดมาตั้งแต่ Supervied Learning แล้ว** — แค่ไม่ได้เรียกชื่อนี้

Neural Network ก็คือการเอา Perceptron หลาย ๆ เซลล์มาต่อกันเป็นชั้น ๆ เท่านั้น

### 2.3 ข้อจำกัดของเซลล์เดียว — ปัญหาที่เส้นตรงแก้ไม่ได้

Perceptron 1 เซลล์ วาดเส้นแบ่งได้แค่ **เส้นตรงเส้นเดียว**

ลองดูข้อมูลรูป "สองพระจันทร์" (make_moons) — สองกลุ่มที่สานกันเป็นรูปโค้ง เส้นตรงเส้นเดียวจะแบ่งได้ไหม?

In [ ]:
# ========== สร้างข้อมูล "สองพระจันทร์" + ลองใช้ Logistic Regression (เส้นตรง) ==========
np.random.seed(42)
X_moon, y_moon = make_moons(n_samples=1000, noise=0.20, random_state=42)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(
    X_moon, y_moon, test_size=0.25, random_state=42, stratify=y_moon)

# ฝึก Logistic Regression = Perceptron 1 เซลล์
logreg = LogisticRegression()
logreg.fit(Xm_tr, ym_tr)
acc_lr = accuracy_score(ym_te, logreg.predict(Xm_te))

# ฟังก์ชันวาด decision boundary (จะใช้ซ้ำหลายรอบ)
def plot_boundary(model, X, y, ax, title, is_torch=False):
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.linspace(x_min,x_max,300), np.linspace(y_min,y_max,300))
    grid = np.c_[xx.ravel(), yy.ravel()]
    if is_torch:
        model.eval()
        with torch.no_grad():
            Z = model(torch.tensor(grid, dtype=torch.float32)).argmax(1).numpy()
    else:
        Z = model.predict(grid)
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
    ax.scatter(X[:,0], X[:,1], c=y, cmap='coolwarm', edgecolors='k', s=18, alpha=0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

fig, ax = plt.subplots(figsize=(6, 5))
plot_boundary(logreg, X_moon, y_moon, ax,
              f'Logistic Regression (Perceptron 1 เซลล์)\nเส้นตรงเส้นเดียว — accuracy = {acc_lr:.3f}')
plt.tight_layout(); plt.show()

print(f"Logistic Regression แบ่งได้ accuracy = {acc_lr:.3f}")
print("สังเกต: เส้นแบ่งเป็นเส้นตรง ตัดผ่ากลางพระจันทร์ทั้งสอง -> มีจุดที่แบ่งผิดเยอะ")

---

## ช่วงที่ 3: ทำไมต้อง "ลึก" + Activation Function

เซลล์เดียวแบ่งสองพระจันทร์ไม่ได้ — แล้วต้องทำยังไง? คำตอบคือ **ซ้อนเซลล์เป็นชั้น ๆ** และใส่ของสำคัญที่ขาดไม่ได้: **Activation Function**

### 3.1 เพิ่มชั้นซ่อน (Hidden Layer) → Multi-Layer Perceptron

ถ้าเซลล์เดียวไม่พอ ก็เอาหลายเซลล์มาต่อกันเป็น **ชั้น** แล้วซ้อนหลายชั้น:

`input layer → hidden layer → output layer`

- **Input layer:** รับ feature เข้ามา (เช่น vibration, temperature)
- **Hidden layer:** ชั้นกลางที่ "คิด" — ยิ่งมีหลายชั้น = ยิ่ง "ลึก" (Deep)
- **Output layer:** ให้คำตอบสุดท้าย

คำว่า **Deep** ใน Deep Learning มาจากตรงนี้ — มี hidden layer หลายชั้น

In [ ]:
# ========== วาดภาพ Multi-Layer Perceptron (MLP) ==========
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axis('off')
layers = [3, 4, 4, 2]   # input=3, hidden 4, hidden 4, output 2
labels = ['Input\n(3 features)', 'Hidden 1\n(4 เซลล์)', 'Hidden 2\n(4 เซลล์)', 'Output\n(Normal/Defect)']
colors = ['#4C72B0', '#DD8452', '#DD8452', '#55A868']
xs = [1, 3.3, 5.6, 7.9]
pos = {}
for li, (n, x, c) in enumerate(zip(layers, xs, colors)):
    ys = np.linspace(3.5, 0.5, n) if n > 1 else [2]
    for ni, y in enumerate(ys):
        ax.add_patch(plt.Circle((x, y), 0.22, color=c, alpha=0.9, zorder=3))
        pos[(li, ni)] = (x, y)
    ax.text(x, 4.1, labels[li], ha='center', fontsize=9, fontweight='bold')
# วาดเส้นเชื่อม
for li in range(len(layers)-1):
    for a in range(layers[li]):
        for b in range(layers[li+1]):
            x1,y1 = pos[(li,a)]; x2,y2 = pos[(li+1,b)]
            ax.plot([x1,x2],[y1,y2], color='gray', alpha=0.3, lw=0.7, zorder=1)
ax.set_xlim(0, 9.2); ax.set_ylim(0, 4.5)
ax.set_title('Multi-Layer Perceptron (MLP): เอา Perceptron หลายเซลล์มาต่อกันเป็นชั้น', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

### 3.2 Intuition หัวใจ: ทำไม "ซ้อนชั้นเฉย ๆ" ไม่พอ

มีกับดักที่หลายคนไม่รู้: **ถ้าซ้อนชั้นเชิงเส้น (Linear) ล้วน ๆ โดยไม่มีอะไรคั่น — มันจะยุบเหลือชั้นเดียว!**

เพราะในทางคณิตศาสตร์: `เส้นตรง ของ เส้นตรง = เส้นตรง`

ลองพิสูจน์ด้วยตัวเลขจริง: เอา Linear 2 ชั้นมาต่อกัน แล้วดูว่ามันเท่ากับ Linear ชั้นเดียวไหม

In [ ]:
# ========== พิสูจน์: Linear + Linear = Linear (ยุบเหลือชั้นเดียว) ==========
np.random.seed(42)
x = np.array([[2.0, 3.0]])           # input 1 ตัวอย่าง 2 feature

# ชั้นที่ 1 (linear): y1 = x @ W1 + b1
W1 = np.random.randn(2, 4)
b1 = np.random.randn(4)
# ชั้นที่ 2 (linear): y2 = y1 @ W2 + b2
W2 = np.random.randn(4, 2)
b2 = np.random.randn(2)

# วิธี A: ส่งผ่านทีละชั้น (ไม่มี activation คั่น)
y1 = x @ W1 + b1
out_two_layers = y1 @ W2 + b2

# วิธี B: ยุบเป็นชั้นเดียว  W_รวม = W1 @ W2 , b_รวม = b1 @ W2 + b2
W_combined = W1 @ W2
b_combined = b1 @ W2 + b2
out_one_layer = x @ W_combined + b_combined

print("ผลลัพธ์จาก 2 ชั้น Linear   :", out_two_layers.round(4))
print("ผลลัพธ์จาก 1 ชั้น (ยุบรวม) :", out_one_layer.round(4))
print("เท่ากันเป๊ะหรือไม่         :", np.allclose(out_two_layers, out_one_layer))
print()
print("สรุป: ซ้อน Linear กี่ชั้นก็เท่ากับชั้นเดียว -> เพิ่มชั้นไปก็ไม่ฉลาดขึ้น")
print("ทางแก้: ใส่ Activation Function (ฟังก์ชันไม่เป็นเส้นตรง) คั่นระหว่างชั้น")

### 3.3 Activation Function — ตัวที่ทำให้ NN "หักโค้ง" ได้

Activation Function คือฟังก์ชัน **ไม่เป็นเส้นตรง** ที่ใส่คั่นหลังแต่ละชั้น

มันทำให้ NN วาดเส้นแบ่งแบบ **โค้ง ซับซ้อน** ได้ ไม่ใช่แค่เส้นตรง

มาดู 3 ตัวที่ใช้บ่อยที่สุด (Softmax จะอธิบายแยก เพราะใช้ที่ชั้น output)

In [ ]:
# ========== วาดกราฟ Activation Function 3 ตัวหลัก ==========
z = np.linspace(-5, 5, 200)
relu = np.maximum(0, z)
sigmoid = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
specs = [
    (relu,    'ReLU', '#C44E52', 'max(0, z)\nค่าลบ -> 0, ค่าบวก -> เท่าเดิม'),
    (sigmoid, 'Sigmoid', '#4C72B0', 'บีบเป็น 0 ถึง 1\nเหมาะกับ output ความน่าจะเป็น'),
    (tanh,    'Tanh', '#55A868', 'บีบเป็น -1 ถึง 1\nเหมือน Sigmoid แต่กลางอยู่ที่ 0'),
]
for ax, (y, name, c, desc) in zip(axes, specs):
    ax.plot(z, y, color=c, lw=2.5)
    ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.text(0.5, -0.25, desc, transform=ax.transAxes, ha='center', fontsize=9, color='#444')
    ax.grid(alpha=0.2)
plt.suptitle('Activation Function 3 ตัวหลัก — ตัวที่ทำให้ NN หักโค้งได้', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

### 3.4 กฎเลือก Activation (จำง่าย ๆ ใช้ได้จริงปี 2026)

| ตำแหน่ง | ใช้ตัวไหน | เหตุผล |
|---------|-----------|--------|
| **ชั้นซ่อน (hidden)** | **ReLU** (ค่าเริ่มต้น) | เร็ว ไม่มีปัญหา gradient หาย เป็นมาตรฐานวงการ |
| ชั้น output — แยก 2 กลุ่ม | Sigmoid | ให้ค่า 0-1 เป็นความน่าจะเป็น |
| ชั้น output — แยกหลายกลุ่ม | Softmax | ให้ความน่าจะเป็นรวมกันได้ 1 |

**กฎทอง:** เริ่มจาก ReLU ในชั้นซ่อนเสมอ — ถ้าเจอปัญหาค่อยลองตัวอื่น (เช่น Leaky ReLU)

**ห้ามใช้ Sigmoid/Tanh ในชั้นซ่อนของเครือข่ายลึก** — เพราะ gradient จะค่อย ๆ หายไป (vanishing gradient) ทำให้เรียนรู้ช้ามาก

### 3.5 พิสูจน์: MLP + ReLU แก้สองพระจันทร์ได้

กลับไปที่ปัญหาสองพระจันทร์ที่ Logistic Regression ทำไม่ได้ — คราวนี้ใช้ MLP ที่มี hidden layer + ReLU ดูว่าต่างกันแค่ไหน

In [ ]:
# ========== MLP + ReLU แก้สองพระจันทร์ — เทียบกับ Logistic ==========
# ใช้ MLPClassifier ของ sklearn ก่อน (ช่วงหน้าจะเขียนเองด้วย PyTorch)
mlp = MLPClassifier(hidden_layer_sizes=(16,), activation='relu',
                    max_iter=2000, random_state=42)
mlp.fit(Xm_tr, ym_tr)
acc_mlp = accuracy_score(ym_te, mlp.predict(Xm_te))

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plot_boundary(logreg, X_moon, y_moon, axes[0],
              f'Logistic (เส้นตรง)\naccuracy = {acc_lr:.3f}')
plot_boundary(mlp, X_moon, y_moon, axes[1],
              f'MLP + ReLU (โค้งได้)\naccuracy = {acc_mlp:.3f}')
plt.suptitle('เซลล์เดียว vs หลายชั้น+ReLU — เห็นความต่างชัดเจน', fontsize=13, fontweight='bold', y=1.0)
plt.tight_layout(); plt.show()

print(f"Logistic (เส้นตรง) : accuracy = {acc_lr:.3f}")
print(f"MLP + ReLU (โค้ง)  : accuracy = {acc_mlp:.3f}")
print(f"ดีขึ้น             : +{(acc_mlp-acc_lr)*100:.1f} จุด")
print()
print("เส้นแบ่งของ MLP โค้งตามรูปพระจันทร์ได้ -> นี่คือพลังของ hidden layer + activation")

---

## ช่วงที่ 4: PyTorch 101 — เครื่องมือสร้าง Neural Network

ที่ผ่านมาเราใช้ MLPClassifier ของ sklearn (ง่ายแต่ปรับแต่งได้น้อย) ตอนนี้ถึงเวลาใช้เครื่องมือจริงของวงการ — **PyTorch**

### 4.1 ทำไมต้อง PyTorch + 3 ส่วนหลัก

**PyTorch** เป็น framework สร้าง Neural Network ที่นิยมที่สุดในวงการวิจัยและอุตสาหกรรม (ปี 2026 เวอร์ชัน 2.x) — ChatGPT, Tesla, Midjourney ล้วนสร้างด้วยเครื่องมือแนวนี้

ข้อดี: เขียนเหมือน Python ปกติ, อ่านง่าย, รันบน GPU ได้, มี community ใหญ่

PyTorch มี 3 ส่วนหลักที่เราต้องรู้จัก:

| ส่วน | คืออะไร | เปรียบเหมือน |
|------|---------|--------------|
| **Tensor** | โครงสร้างข้อมูลหลัก | NumPy array ที่วิ่งบน GPU ได้ |
| **Autograd** | คำนวณ gradient อัตโนมัติ | สมุดจดที่จำทุกการคำนวณไว้ย้อนกลับได้ |
| **nn.Module** | โครงสร้างโมเดล | แม่แบบสร้าง Neural Network |

มาทำความรู้จักทีละตัว

### 4.2 Tensor — ก้อนข้อมูลของ PyTorch

Tensor คือ NumPy array เวอร์ชัน PyTorch — เก็บตัวเลขเป็นตาราง บวกลบคูณได้เหมือนกัน แต่พิเศษกว่าตรงวิ่งบน GPU ได้และจำการคำนวณเพื่อหา gradient ได้

In [ ]:
# ========== ทำความรู้จัก Tensor ==========
# สร้าง tensor จาก list
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print("Tensor a:")
print(a)
print("ขนาด (shape):", a.shape)
print()

# คำนวณได้เหมือน numpy
b = torch.tensor([[10.0, 20.0], [30.0, 40.0]])
print("a + b =")
print(a + b)
print()
print("a @ b (คูณเมทริกซ์) =")
print(a @ b)
print()

# แปลงไป-กลับกับ numpy ได้ง่าย
np_array = np.array([1.5, 2.5, 3.5])
t = torch.tensor(np_array, dtype=torch.float32)
print("จาก numpy เป็น tensor:", t)
print("จาก tensor กลับเป็น numpy:", t.numpy())

### 4.3 Autograd — คำนวณ Gradient อัตโนมัติ

**Gradient Descent** — การหาความชันเพื่อ "ลงเขาตาบอด" ทีละก้าวไปหาจุดต่ำสุดของ loss

PyTorch มี **Autograd** ที่คำนวณ gradient ให้ **อัตโนมัติ** — แค่บอกว่าอยากติดตามตัวไหน แล้วเรียก `.backward()`

In [ ]:
# ========== Autograd: คำนวณ gradient อัตโนมัติ ==========
# ตัวอย่างง่าย: y = x^2  ที่จุด x = 3
# เราคำนวณมือได้ว่า dy/dx = 2x = 2(3) = 6
x = torch.tensor(3.0, requires_grad=True)   # requires_grad = ขอติดตามตัวนี้
y = x ** 2

y.backward()        # คำนวณ gradient ย้อนกลับ
print("ที่จุด x = 3")
print("y = x^2 =", y.item())
print("gradient dy/dx (PyTorch คำนวณให้):", x.grad.item())
print("คำตอบที่ถูก (2x = 2*3)            : 6.0")
print()
print("Autograd คำนวณ gradient ให้อัตโนมัติ -> นี่คือหัวใจที่ทำให้ฝึก NN ขนาดยักษ์ได้")

### 4.4 nn.Module — แม่แบบสร้าง Neural Network

`nn.Module` คือแม่แบบที่เราสืบทอด (inherit) มาสร้างโมเดล มี 2 ส่วนที่ต้องเขียน:

1. `__init__` — ประกาศว่ามีชั้น (layer) อะไรบ้าง
2. `forward` — บอกว่าข้อมูลไหลผ่านชั้นยังไง (forward pass)

โครงสร้างนี้ใช้ได้กับทุกขนาด — ตั้งแต่ NN เล็ก ๆ จนถึง Transformer ของ ChatGPT ก็ใช้แม่แบบเดียวกันนี้

In [ ]:
# ========== สร้าง Neural Network ด้วย nn.Module ==========
class SimpleNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        # ประกาศชั้น: input -> hidden(16) -> output(2 กลุ่ม)
        self.fc1 = nn.Linear(n_features, 16)   # ชั้นแรก
        self.relu = nn.ReLU()                  # activation
        self.fc2 = nn.Linear(16, 2)            # ชั้น output (2 กลุ่ม: Normal/Defect)

    def forward(self, x):
        # บอกว่าข้อมูลไหลยังไง: ผ่าน fc1 -> relu -> fc2
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)         # ส่งออกเป็น "raw score" (logits) -- ยังไม่ผ่าน softmax
        return x

# สร้างโมเดลสำหรับข้อมูล 2 feature (สองพระจันทร์)
model_demo = SimpleNN(n_features=2)
print(model_demo)
print()
# นับจำนวน parameter (weight + bias ที่โมเดลต้องเรียนรู้)
n_params = sum(p.numel() for p in model_demo.parameters())
print(f"จำนวน parameter ที่ต้องเรียนรู้: {n_params} ตัว")

---

## ช่วงที่ 5: Training Loop — หัวใจการเรียนรู้

มีโมเดลแล้ว แต่มันยัง "โง่" อยู่ (weight สุ่มมา) เราต้อง **ฝึก** มันให้เรียนรู้จากข้อมูล — นี่คือหัวใจของ Deep Learning

### 5.1 ห้าบรรทัดทอง — Training Loop ที่ใช้ซ้ำได้ทั้ง Phase 3

การฝึก NN คือทำ 5 ขั้นนี้ซ้ำ ๆ หลายรอบ (แต่ละรอบเรียก epoch):

```python
optimizer.zero_grad()        # 1. ล้าง gradient เก่า
output = model(X)            # 2. forward: ทำนาย
loss = criterion(output, y) # 3. คำนวณว่าผิดแค่ไหน
loss.backward()             # 4. backprop: หา gradient (autograd)
optimizer.step()            # 5. ปรับ weight ลงเขาทีละก้าว
```

ห้าบรรทัดนี้ไม่เปลี่ยนเลย ไม่ว่าโมเดลจะเล็กหรือใหญ่แค่ไหน

### 5.2 Loss Function + คำเตือนสำคัญที่มือใหม่พลาดบ่อยที่สุด

**Loss Function** = ตัววัดว่าโมเดลทำนายผิดแค่ไหน (ยิ่งน้อยยิ่งดี)

สำหรับงาน Classification เราใช้ `nn.CrossEntropyLoss`

**คำเตือนอันดับ 1 ที่มือใหม่พลาด:** `CrossEntropyLoss` ของ PyTorch **ทำ Softmax ให้ในตัวอยู่แล้ว** ดังนั้น:

- ชั้น output ของโมเดลต้องส่ง **raw score (logits)** ออกมาตรง ๆ
- **ห้ามใส่ `nn.Softmax()` ในโมเดลเอง** — ถ้าใส่ = Softmax ซ้อน 2 ที = โมเดลเรียนรู้เพี้ยน

(ตอน inference ถ้าอยากได้ความน่าจะเป็น ค่อยเรียก `torch.softmax()` แยกทีหลัง)

### 5.3 Optimizer — ตัวปรับ weight (SGD → Adam)

**Optimizer** คือตัวที่เอา gradient ไปปรับ weight จริง ๆ

| Optimizer | คือ | เมื่อไหร่ใช้ |
|-----------|-----|-------------|
| **SGD** | Stochastic Gradient Descent — ลงเขาตรง ๆ (Week 3 เป๊ะ) | เข้าใจง่าย เห็นภาพ |
| **Adam** | SGD เวอร์ชันฉลาด ปรับก้าวอัตโนมัติ | ค่าเริ่มต้นที่ทุกคนใช้จริง — ลู่เข้าเร็วกว่า |

ใช้ **Adam** เพราะฝึกได้เร็วและนิ่งกว่า แต่ให้จำไว้ว่าแก่นมันคือ Gradient Descent

### 5.4 Workshop — ฝึก Neural Network ตัวแรกด้วย PyTorch (บนสองพระจันทร์)

มาประกอบทุกชิ้นเข้าด้วยกัน ฝึก NN ของเราเองบนข้อมูลสองพระจันทร์

In [ ]:
# ========== Step 1: เตรียมข้อมูล — scale + แปลงเป็น Tensor ==========
# NN ไวต่อสเกลข้อมูล (ต่างจาก Random Forest) -> ต้อง scale ก่อนเสมอ
scaler_moon = StandardScaler()
Xm_tr_s = scaler_moon.fit_transform(Xm_tr)
Xm_te_s = scaler_moon.transform(Xm_te)

# แปลงเป็น tensor: X เป็น float, y เป็น long (integer สำหรับ class)
Xtr_t = torch.tensor(Xm_tr_s, dtype=torch.float32)
ytr_t = torch.tensor(ym_tr, dtype=torch.long)
Xte_t = torch.tensor(Xm_te_s, dtype=torch.float32)
yte_t = torch.tensor(ym_te, dtype=torch.long)

print("ข้อมูลพร้อมเป็น Tensor แล้ว")
print("X train shape:", Xtr_t.shape, "| y train shape:", ytr_t.shape)

In [ ]:
# ========== Step 2: สร้างโมเดล + loss + optimizer ==========
torch.manual_seed(42)
model = SimpleNN(n_features=2)

criterion = nn.CrossEntropyLoss()                       # loss (มี softmax ในตัว)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)  # Adam = SGD ฉลาด

print("โมเดล + loss + optimizer พร้อมแล้ว")

In [ ]:
# ========== Step 3: Training Loop — 5 บรรทัดทอง ==========
n_epochs = 200
loss_history = []

for epoch in range(n_epochs):
    optimizer.zero_grad()              # 1. ล้าง gradient เก่า
    output = model(Xtr_t)              # 2. forward
    loss = criterion(output, ytr_t)    # 3. คำนวณ loss
    loss.backward()                    # 4. backprop (autograd หา gradient)
    optimizer.step()                   # 5. ปรับ weight

    loss_history.append(loss.item())
    if (epoch+1) % 40 == 0:
        print(f"Epoch {epoch+1:3d}/{n_epochs} | loss = {loss.item():.4f}")

print()
print(f"ฝึกเสร็จ — loss ลดจาก {loss_history[0]:.4f} เหลือ {loss_history[-1]:.4f}")

In [ ]:
# ========== Step 4: วาดกราฟ loss ลดลง (โมเดลกำลังเรียนรู้) ==========
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(loss_history, color='#C44E52', lw=2)
ax.set_xlabel('Epoch (รอบการฝึก)', fontsize=11)
ax.set_ylabel('Loss (ยิ่งน้อยยิ่งดี)', fontsize=11)
ax.set_title('Loss ลดลงเรื่อย ๆ — โมเดลค่อย ๆ ฉลาดขึ้นทุก epoch', fontsize=12, fontweight='bold')
ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()
print("กราฟ loss ดิ่งลง = โมเดลกำลัง 'ลงเขา' หาจุดที่ทำนายผิดน้อยที่สุด (Gradient Descent จาก Week 3)")

In [ ]:
# ========== Step 5: ดูผล — decision boundary + accuracy ==========
model.eval()
with torch.no_grad():
    logits = model(Xte_t)
    proba = torch.softmax(logits, dim=1)        # ตอน inference ค่อยใส่ softmax เอง
    pred = proba.argmax(dim=1).numpy()
acc_torch = accuracy_score(ym_te, pred)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
plot_boundary(logreg, X_moon, y_moon, axes[0], f'Logistic \nacc = {acc_lr:.3f}')
# โมเดล PyTorch ของเรา (ต้อง scale grid ด้วย ก่อนส่งเข้า)
class WrappedModel:
    def predict(self, grid):
        gs = scaler_moon.transform(grid)
        with torch.no_grad():
            return model(torch.tensor(gs, dtype=torch.float32)).argmax(1).numpy()
plot_boundary(WrappedModel(), X_moon, y_moon, axes[1],
              f'Neural Net ของเรา (PyTorch)\nacc = {acc_torch:.3f}')
plt.suptitle('Neural Network ตัวแรกที่เราเขียนเอง — แก้สองพระจันทร์ได้!', fontsize=13, fontweight='bold', y=1.0)
plt.tight_layout(); plt.show()

print(f"Neural Network ของเรา (เขียนเองด้วย PyTorch): accuracy = {acc_torch:.3f}")
print("เส้นแบ่งโค้งตามพระจันทร์ได้ -> เราสร้าง NN ที่ทำงานจริงสำเร็จแล้ว")

### 5.5 สรุปช่วงที่ 5

เราสร้างและฝึก Neural Network ตัวแรกด้วย PyTorch สำเร็จ ด้วย workflow มาตรฐาน:

1. เตรียมข้อมูล (scale + แปลงเป็น Tensor)
2. สร้างโมเดล (nn.Module) + loss (CrossEntropyLoss) + optimizer (Adam)
3. Training loop — 5 บรรทัดทอง วนหลาย epoch
4. ดู loss ลดลง + วัดผล

**workflow นี้ใช้ซ้ำได้ตลอด Phase 3** — Week 10 (CNN) และ Week 11 (NLP) ก็ใช้โครงเดียวกันนี้

---

## ช่วงที่ 6: Mini Project — Neural Network ปะทะ Random Forest บนงาน CNC จริง

ถึงเวลาเฉลยปริศนาเปิดเรื่อง: เอา Neural Network ที่เราเพิ่งสร้างเป็น มาสู้กับ Random Forest บน **ข้อมูล ชุดเดิม** แล้วดูว่าใครชนะ

### 6.1 สร้าง CNC Dataset (ชุดเดียวกับ Week 8)

ใช้ข้อมูลโรงงาน CNC : 1,500 ชิ้น, defect 12%, มีทั้ง numeric (vibration, temperature...) และ categorical (machine_id, shift, material)

In [ ]:
# ========== สร้าง CNC Dataset v2 (เหมือน Week 8 เป๊ะ) ==========
np.random.seed(42)
n_total = 1500
n_defect = int(n_total * 0.12)
n_normal = n_total - n_defect
machines = ['M01','M02','M03','M04','M05']
shifts = ['เช้า','บ่าย','ดึก']
materials = ['Steel','Aluminum','Brass']

normal_data = pd.DataFrame({
    'vibration':      np.random.normal(0.8, 0.25, n_normal).clip(0.2, 2.0),
    'temperature':    np.random.normal(65, 4, n_normal).clip(50, 80),
    'current':        np.random.normal(10, 1.5, n_normal).clip(5, 15),
    'dimension_diff': np.abs(np.random.normal(0, 0.03, n_normal)).clip(0, 0.15),
    'surface_score':  np.random.normal(90, 4, n_normal).clip(75, 100),
    'machine_id':     np.random.choice(machines, n_normal, p=[.10,.20,.25,.25,.20]),
    'shift':          np.random.choice(shifts, n_normal, p=[.40,.35,.25]),
    'material':       np.random.choice(materials, n_normal, p=[.45,.35,.20]),
    'is_defect':      0})
defect_data = pd.DataFrame({
    'vibration':      np.random.normal(1.8, 0.5, n_defect).clip(0.5, 4.0),
    'temperature':    np.random.normal(75, 6, n_defect).clip(55, 90),
    'current':        np.random.normal(12, 2.5, n_defect).clip(5, 18),
    'dimension_diff': np.abs(np.random.normal(0, 0.10, n_defect)).clip(0, 0.30),
    'surface_score':  np.random.normal(75, 8, n_defect).clip(50, 95),
    'machine_id':     np.random.choice(machines, n_defect, p=[.40,.25,.15,.10,.10]),
    'shift':          np.random.choice(shifts, n_defect, p=[.20,.30,.50]),
    'material':       np.random.choice(materials, n_defect, p=[.50,.30,.20]),
    'is_defect':      1})

df = pd.concat([normal_data, defect_data], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"CNC Dataset พร้อม: {df.shape[0]} ชิ้น")
print(f"  Normal: {(df['is_defect']==0).sum()} | Defect: {(df['is_defect']==1).sum()}")
df.head()

### 6.2 เตรียมข้อมูลด้วย Pipeline

ใช้ ColumnTransformer + Pipeline  จัดการ numeric (scale) + categorical (one-hot) พร้อมกัน

**ข้อสังเกตสำคัญ:** Neural Network **ต้อง scale ข้อมูล** (ไวต่อสเกล) ส่วน Random Forest ไม่จำเป็น — แต่เราใช้ Pipeline เดียวกันเพื่อเทียบกันยุติธรรม

In [ ]:
# ========== แยก feature + เตรียม preprocessor ==========
num_features = ['vibration','temperature','current','dimension_diff','surface_score']
cat_features = ['machine_id','shift','material']
X = df[num_features + cat_features]
y = df['is_defect'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_features),
])
print("Train:", X_train.shape[0], "ชิ้น | Test:", X_test.shape[0], "ชิ้น")

### 6.3 ผู้ท้าชิงที่ 1: Random Forest

In [ ]:
# ========== Random Forest — แชมป์เก่า ==========
rf_pipeline = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42,
                                   class_weight='balanced'))
])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_f1 = f1_score(y_test, rf_pred)
rf_acc = accuracy_score(y_test, rf_pred)
print(f"Random Forest -> F1 = {rf_f1:.3f} | Accuracy = {rf_acc:.3f}")

### 6.4 ผู้ท้าชิงที่ 2: Neural Network (ที่เราสร้างเอง)

ใช้ preprocessor ตัวเดียวกัน แปลงข้อมูลแล้วฝึก NN ด้วย workflow PyTorch ของช่วงที่ 5

In [ ]:
# ========== เตรียมข้อมูลสำหรับ NN ==========
preprocessor.fit(X_train)
X_train_p = np.asarray(preprocessor.transform(X_train))
X_test_p  = np.asarray(preprocessor.transform(X_test))
n_feat = X_train_p.shape[1]   # จำนวน feature หลัง one-hot ขยาย

Xtr_t = torch.tensor(X_train_p, dtype=torch.float32)
ytr_t = torch.tensor(y_train, dtype=torch.long)
Xte_t = torch.tensor(X_test_p, dtype=torch.float32)
print(f"หลัง preprocessing: {n_feat} feature (numeric 5 + categorical ที่ one-hot ขยาย)")

In [ ]:
# ========== สร้าง NN สำหรับ CNC (ลึกขึ้นนิด) + ฝึก ==========
torch.manual_seed(42)
np.random.seed(42)

class QCNet(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 32), nn.ReLU(),
            nn.Linear(32, 16),         nn.ReLU(),
            nn.Linear(16, 2),                       # output 2 กลุ่ม (raw logits)
        )
    def forward(self, x):
        return self.net(x)

qc_model = QCNet(n_feat)
# defect มีน้อย (12%) -> ถ่วงน้ำหนัก class defect ให้โมเดลใส่ใจมากขึ้น
criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 7.0]))
optimizer = torch.optim.Adam(qc_model.parameters(), lr=0.01)

for epoch in range(300):
    optimizer.zero_grad()
    loss = criterion(qc_model(Xtr_t), ytr_t)
    loss.backward()
    optimizer.step()
    if (epoch+1) % 75 == 0:
        print(f"Epoch {epoch+1:3d} | loss = {loss.item():.4f}")

qc_model.eval()
with torch.no_grad():
    nn_pred = qc_model(Xte_t).argmax(1).numpy()
nn_f1 = f1_score(y_test, nn_pred)
nn_acc = accuracy_score(y_test, nn_pred)
print()
print(f"Neural Network -> F1 = {nn_f1:.3f} | Accuracy = {nn_acc:.3f}")

In [ ]:
# ========== เทียบหน้าต่อหน้า (seed เดียว) ==========
print("="*45)
print(f"{'โมเดล':<20}{'F1':>10}{'Accuracy':>12}")
print("-"*45)
print(f"{'Random Forest':<20}{rf_f1:>10.3f}{rf_acc:>12.3f}")
print(f"{'Neural Network':<20}{nn_f1:>10.3f}{nn_acc:>12.3f}")
print("="*45)
print()
print("seed เดียวสรุปไม่ได้ -> ต้องรันหลาย seed ก่อนตัดสิน (วิธีคิดแบบวิทยาศาสตร์)")

### 6.5 อย่าด่วนสรุปจาก seed เดียว — รันหลายรอบเทียบกัน

ผลจาก seed เดียวอาจหลอกตาได้ (บังเอิญ) วิธีที่ถูกต้องคือ **รันหลาย seed แล้วดูค่าเฉลี่ย** — นี่คือวิธีคิดเชิงวิทยาศาสตร์ที่ต้องติดตัวไป

In [ ]:
# ========== รัน 5 seed: RF vs NN — ใครชนะจริง? ==========
def build_cnc(seed):
    np.random.seed(seed)
    nd = int(1500*0.12); nn_ = 1500-nd
    M=['M01','M02','M03','M04','M05']; S=['เช้า','บ่าย','ดึก']; MA=['Steel','Aluminum','Brass']
    nrm = pd.DataFrame({'vibration':np.random.normal(0.8,0.25,nn_).clip(0.2,2.0),'temperature':np.random.normal(65,4,nn_).clip(50,80),'current':np.random.normal(10,1.5,nn_).clip(5,15),'dimension_diff':np.abs(np.random.normal(0,0.03,nn_)).clip(0,0.15),'surface_score':np.random.normal(90,4,nn_).clip(75,100),'machine_id':np.random.choice(M,nn_,p=[.10,.20,.25,.25,.20]),'shift':np.random.choice(S,nn_,p=[.40,.35,.25]),'material':np.random.choice(MA,nn_,p=[.45,.35,.20]),'is_defect':0})
    dfc = pd.DataFrame({'vibration':np.random.normal(1.8,0.5,nd).clip(0.5,4.0),'temperature':np.random.normal(75,6,nd).clip(55,90),'current':np.random.normal(12,2.5,nd).clip(5,18),'dimension_diff':np.abs(np.random.normal(0,0.10,nd)).clip(0,0.30),'surface_score':np.random.normal(75,8,nd).clip(50,95),'machine_id':np.random.choice(M,nd,p=[.40,.25,.15,.10,.10]),'shift':np.random.choice(S,nd,p=[.20,.30,.50]),'material':np.random.choice(MA,nd,p=[.50,.30,.20]),'is_defect':1})
    return pd.concat([nrm,dfc],ignore_index=True).sample(frac=1,random_state=seed).reset_index(drop=True)

seeds = [42, 1, 7, 123, 2024]
rf_list, nn_list = [], []
for sd in seeds:
    d = build_cnc(sd); Xs = d[num_features+cat_features]; ys = d['is_defect'].values
    Xtr_, Xte_, ytr_, yte_ = train_test_split(Xs, ys, test_size=0.2, random_state=sd, stratify=ys)
    pre = ColumnTransformer([('num',Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]),num_features),('cat',Pipeline([('i',SimpleImputer(strategy='most_frequent')),('o',OneHotEncoder(handle_unknown='ignore'))]),cat_features)])
    rf = Pipeline([('p',pre),('c',RandomForestClassifier(n_estimators=200,random_state=sd,class_weight='balanced'))]).fit(Xtr_,ytr_)
    rf_list.append(f1_score(yte_, rf.predict(Xte_)))
    pre.fit(Xtr_); tr = np.asarray(pre.transform(Xtr_)); te = np.asarray(pre.transform(Xte_))
    torch.manual_seed(sd); np.random.seed(sd)
    m = nn.Sequential(nn.Linear(tr.shape[1],32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,2))
    cr = nn.CrossEntropyLoss(weight=torch.tensor([1.0,7.0])); op = torch.optim.Adam(m.parameters(),lr=0.01)
    Xt = torch.tensor(tr,dtype=torch.float32); yt = torch.tensor(ytr_,dtype=torch.long); Xv = torch.tensor(te,dtype=torch.float32)
    for e in range(300): op.zero_grad(); l=cr(m(Xt),yt); l.backward(); op.step()
    m.eval()
    with torch.no_grad(): nn_list.append(f1_score(yte_, m(Xv).argmax(1).numpy()))

rf_arr, nn_arr = np.array(rf_list), np.array(nn_list)
print(f"{'seed':<8}{'RF (F1)':>10}{'NN (F1)':>10}")
print("-"*28)
for sd, r, n in zip(seeds, rf_arr, nn_arr):
    print(f"{sd:<8}{r:>10.3f}{n:>10.3f}")
print("-"*28)
print(f"{'เฉลี่ย':<8}{rf_arr.mean():>10.3f}{nn_arr.mean():>10.3f}")
print(f"{'ส่วนเบี่ยงเบน':<8}{rf_arr.std():>10.3f}{nn_arr.std():>10.3f}")

In [ ]:
# ========== วาด bar chart เทียบ RF vs NN ทุก seed ==========
fig, ax = plt.subplots(figsize=(9, 5))
xpos = np.arange(len(seeds)); w = 0.35
ax.bar(xpos - w/2, rf_arr, w, label='Random Forest', color='#55A868', alpha=0.85)
ax.bar(xpos + w/2, nn_arr, w, label='Neural Network', color='#C44E52', alpha=0.85)
ax.axhline(rf_arr.mean(), color='#55A868', ls='--', lw=1, alpha=0.7)
ax.axhline(nn_arr.mean(), color='#C44E52', ls='--', lw=1, alpha=0.7)
ax.set_xticks(xpos); ax.set_xticklabels([f'seed {s}' for s in seeds])
ax.set_ylabel('F1-score', fontsize=11)
ax.set_ylim(0.85, 1.01)
ax.set_title('Random Forest vs Neural Network บน CNC (5 seed)\nสูสีกันมาก — สลับกันชนะ', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.25)
plt.tight_layout(); plt.show()
print(f"เฉลี่ย: RF = {rf_arr.mean():.3f} | NN = {nn_arr.mean():.3f} | ต่างกันแค่ {abs(nn_arr.mean()-rf_arr.mean()):.3f}")

### 6.6 เฉลยปริศนา — ความจริงที่หลายคนเข้าใจผิด

ผลที่ได้ (รันเองได้เลย ไม่ได้แต่ง):
- Random Forest กับ Neural Network **สูสีกันมาก** — ต่างกันแค่ราว 1 จุด F1 และ **สลับกันชนะ** ในแต่ละ seed
- บนข้อมูลตารางแบบนี้ **Deep Learning ไม่ได้ดีกว่าเสมอไป**

**ความจริงระดับงานวิจัย (ยืนยันถึงปี 2026):**
> บน **ข้อมูลตาราง (tabular)** ขนาดกลาง — โมเดลกลุ่มต้นไม้อย่าง **Random Forest / XGBoost มักจะชนะหรือเสมอ** Neural Network

เหตุผล: ข้อมูลตารางจริงมักมี feature ที่ไม่เกี่ยว + เส้นแบ่งที่ "หักศอก" ซึ่งต้นไม้ถนัด ส่วน NN ถนัดเส้นแบ่งเรียบ ๆ

(หมายเหตุตรง ๆ: dataset เราเป็นข้อมูลสังเคราะห์จากการสุ่มแบบ Gaussian ซึ่งเส้นแบ่งค่อนข้างเรียบ จึงเข้าทาง NN นิดหน่อย — ของจริงในโรงงานจะมี noise และกฎหักศอกมากกว่านี้ Random Forest จะยิ่งได้เปรียบ)

**แล้วเรียน Neural Network ไปทำไม?** เพราะมันคือเครื่องมือเดียวที่ทำสิ่งที่ Random Forest **ทำไม่ได้เลย**:
- **มองภาพ** ชิ้นงาน QC จากกล้อง
- **อ่านเอกสาร/คู่มือ** และตอบคำถาม
- ขับเคลื่อน ChatGPT, Midjourney, Tesla

**บทเรียนที่ต้องจำ:** เลือกเครื่องมือตามงานและข้อมูล — ไม่ใช่ตามความเท่ บนตาราง Random Forest ยังเป็นตัวเลือกที่ดีและคุ้มค่ากว่า (เร็วกว่า ไม่ต้อง scale ไม่ต้อง tune เยอะ)

---

## จบการสร้าง Neural Network ตัวแรกสำเร็จแล้ว

**สิ่งที่ทำได้วันนี้:**
- เข้าใจ Neural Network ตั้งแต่เซลล์เดียว (Perceptron) จนถึงหลายชั้น (Deep)
- รู้ว่าทำไมต้องมี Activation Function
- สร้างและฝึก Neural Network ด้วย PyTorch ด้วยตัวเอง
- ค้นพบความจริง: บนตาราง Random Forest ยังสูสี — NN ทำงานได้ดีตอนเจอภาพและภาษา
